In [47]:
# Ensure RNN runs log to the assignment-specific folder
from logging_config import ensure_logdir, get_logdir
LOGDIR = ensure_logdir()
print('RNN LOGDIR set to', LOGDIR)


RNN LOGDIR set to 3-hypertuning-rnn/modellogs


In [48]:
from pathlib import Path
import numpy as np
import torch
from typing import List
from torch.nn.utils.rnn import pad_sequence
from mltrainer import rnn_models, Trainer
from torch import optim

from mads_datasets import datatools
import mltrainer
mltrainer.__version__

'0.2.5'

In [49]:
# SimpleCNN baseline setup (ready to run)
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import BasePreprocessor
from mltrainer import TrainerSettings, ReportTypes
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from mltrainer import metrics
from pathlib import Path

# create datastreamers (batchsize 128)
factory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
pre = BasePreprocessor()
streamers = factory.create_datastreamer(batchsize=128, preprocessor=pre)
train_stream = streamers['train'].stream()
valid_stream = streamers['valid'].stream()
train_steps = len(streamers['train'])
valid_steps = len(streamers['valid'])
print('Prepared datastreamers: train_steps=', train_steps, 'valid_steps=', valid_steps)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,32,3,padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64*7*7,128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128,10)
    def forward(self,x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

# instantiate model and trainer (not executed)
model = SimpleCNN()
accuracy = metrics.Accuracy()
loss_fn = nn.CrossEntropyLoss()
LOGDIR = Path('3-hypertuning-rnn') / 'modellogs'
LOGDIR.mkdir(parents=True, exist_ok=True)
settings = TrainerSettings(
    epochs=10,
    metrics=[accuracy],
    logdir=str(LOGDIR),
    train_steps=train_steps,
    valid_steps=valid_steps,
    reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
)
trainer = Trainer(
    model=model,
    settings=settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=train_stream,
    validdataloader=valid_stream,
    scheduler=None,
)

print('Setup complete. Run `trainer.loop()` to start training in this kernel.')

2026-06-14 15:07:55.397 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/fblaak/.cache/mads_datasets/fashionmnist
2026-06-14 15:07:55.399 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/fblaak/.cache/mads_datasets/fashionmnist/fashionmnist.pt
2026-06-14 15:07:55.471 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to 3-hypertuning-rnn/modellogs/20260614-150755
2026-06-14 15:07:55.472 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


Prepared datastreamers: train_steps= 468 valid_steps= 78
Setup complete. Run `trainer.loop()` to start training in this kernel.


# 1 Iterators
We will be using an interesting dataset. [link](https://tev.fbk.eu/resources/smartwatch)

From the site:
> The SmartWatch Gestures Dataset has been collected to evaluate several gesture recognition algorithms for interacting with mobile applications using arm gestures. Eight different users performed twenty repetitions of twenty different gestures, for a total of 3200 sequences. Each sequence contains acceleration data from the 3-axis accelerometer of a first generation Sony SmartWatch™, as well as timestamps from the different clock sources available on an Android device. The smartwatch was worn on the user's right wrist. 


In [50]:
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import PaddedPreprocessor
preprocessor = PaddedPreprocessor()

gesturesdatasetfactory = DatasetFactoryProvider.create_factory(DatasetType.GESTURES)
streamers = gesturesdatasetfactory.create_datastreamer(batchsize=32, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]

2026-06-14 15:07:55.481 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/fblaak/.cache/mads_datasets/gestures
100%|██████████| 651/651 [00:00<00:00, 4220.93it/s]


In [51]:
len(train), len(valid)

(81, 20)

In [52]:
trainstreamer = train.stream()
validstreamer = valid.stream()
x, y = next(iter(trainstreamer))
x.shape, y

(torch.Size([32, 31, 3]),
 tensor([19, 14,  0,  2,  0,  6,  9, 15, 13,  3, 18,  4,  6, 15,  5, 15,  1, 14,
          7,  3,  5, 12,  5, 16, 17,  3,  2, 11,  2,  5,  7, 14]))

Can you make sense of the shape?
What does it mean that the shapes are sometimes (32, 27, 3), but a second time might look like (32, 30, 3)? In other words, the second (or first, if you insist on starting at 0) dimension changes. Why is that? How does the model handle this? Do you think this is already padded, or still has to be padded?


# 2 Excercises
Lets test a basemodel, and try to improve upon that.

Fill the gestures.gin file with relevant settings for `input_size`, `hidden_size`, `num_layers` and `horizon` (which, in our case, will be the number of classes...)

As a rule of thumbs: start lower than you expect to need!

In [53]:
from mltrainer import TrainerSettings, ReportTypes
from mltrainer.metrics import Accuracy

accuracy = Accuracy()


In [54]:
model = rnn_models.BaseRNN(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    horizon=20,
)

Test the model. What is the output shape you need? Remember, we are doing classification!

In [55]:
yhat = model(x)
yhat.shape

torch.Size([32, 20])

Test the accuracy

In [56]:
accuracy(y, yhat)

0.03125

What do you think of the accuracy? What would you expect from blind guessing?

Check shape of `y` and `yhat`

In [57]:
yhat.shape, y.shape

(torch.Size([32, 20]), torch.Size([32]))

And look at the output of yhat

In [58]:
yhat[0]

tensor([ 0.0048, -0.0084, -0.1790,  0.0790, -0.0577, -0.1761, -0.0613,  0.1864,
        -0.0007,  0.0946, -0.0430, -0.1379, -0.0242,  0.2047,  0.0927,  0.1863,
         0.0062,  0.1729,  0.1645, -0.0598], grad_fn=<SelectBackward0>)

Does this make sense to you? If you are unclear, go back to the classification problem with the MNIST, where we had 10 classes.

We have a classification problem, so we need Cross Entropy Loss.
Remember, [this has a softmax built in](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) 

In [59]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(yhat, y)
loss

tensor(3.0197, grad_fn=<NllLossBackward0>)

In [60]:
import torch
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

# on my mac, at least for the BaseRNN model, mps does not speed up training
# probably because the overhead of copying the data to the GPU is too high
# so i override the device to cpu
device = "cpu"
# however, it might speed up training for larger models, with more parameters

using cpu


Set up the settings for the trainer and the different types of logging you want

In [61]:
settings = TrainerSettings(
    epochs=3, # increase this to about 100 for training
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    earlystop_kwargs = {
        "save": False, # save every best model, and restore the best one
        "verbose": True,
        "patience": 5, # number of epochs with no improvement after which training will be stopped
        "delta": 0.0, # minimum change to be considered an improvement
    }
)
settings

epochs: 3
metrics: [Accuracy]
logdir: gestures
train_steps: 81
valid_steps: 20
reporttypes: [<ReportTypes.TOML: 'TOML'>, <ReportTypes.TENSORBOARD: 'TENSORBOARD'>, <ReportTypes.MLFLOW: 'MLFLOW'>]
optimizer_kwargs: {'lr': 0.001, 'weight_decay': 1e-05}
scheduler_kwargs: {'factor': 0.5, 'patience': 5}
earlystop_kwargs: {'save': False, 'verbose': True, 'patience': 5, 'delta': 0.0}

In [62]:
import torch.nn as nn
import torch
from torch import Tensor
from dataclasses import dataclass

@dataclass
class ModelConfig:
    input_size: int
    hidden_size: int
    num_layers: int
    output_size: int
    dropout: float = 0.0

class GRUmodel(nn.Module):
    def __init__(
        self,
        config,
    ) -> None:
        super().__init__()
        self.config = config
        self.rnn = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            dropout=config.dropout,
            batch_first=True,
            num_layers=config.num_layers,
        )
        self.linear = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        x, _ = self.rnn(x)
        last_step = x[:, -1, :]
        yhat = self.linear(last_step)
        return yhat

In [63]:
config = ModelConfig(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    output_size=20,
    dropout=0.0,
)


In [64]:
import mlflow
from datetime import datetime

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("gestures")
modeldir = Path("gestures").resolve()
if not modeldir.exists():
    modeldir.mkdir(parents=True)

with mlflow.start_run():
    mlflow.set_tag("model", "modelname-here")
    mlflow.set_tag("dev", "your-name-here")
    config = ModelConfig(
        input_size=3,
        hidden_size=64,
        num_layers=1,
        output_size=20,
        dropout=0.1,
    )

    model = GRUmodel(
        config=config,
    )

    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=loss_fn,
        optimizer=optim.Adam,
        traindataloader=trainstreamer,
        validdataloader=validstreamer,
        scheduler=optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
    )
    trainer.loop()

    if not settings.earlystop_kwargs["save"]:
        tag = datetime.now().strftime("%Y%m%d-%H%M-")
        modelpath = modeldir / (tag + "model.pt")
        torch.save(model, modelpath)

2026-06-14 15:07:56.734 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures/20260614-150756
2026-06-14 15:07:56.735 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 81/81 [00:00<00:00, 83.40it/s]
2026-06-14 15:07:57.842 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.8897 test 2.5376 metric ['0.1109']
100%|██████████| 81/81 [00:00<00:00, 89.00it/s]
2026-06-14 15:07:58.892 | INFO     | mltrainer.trainer:report:209 - Epoch 1 train 2.3584 test 2.3434 metric ['0.1859']
100%|██████████| 81/81 [00:00<00:00, 89.99it/s]
2026-06-14 15:07:59.942 | INFO     | mltrainer.trainer:report:209 - Epoch 2 train 2.1691 test 2.1093 metric ['0.2297']
100%|██████████| 3/3 [00:03<00:00,  1.07s/it]


Try to update the code above by changing the hyperparameters.
    
To discern between the changes, also modify the tag mlflow.set_tag("model", "new-tag-here") where you add
a new tag of your choice. This way you can keep the models apart.

In [65]:
trainer.loop() # if you want to pick up training, loop will continue from the last epoch

100%|██████████| 81/81 [00:00<00:00, 86.47it/s]
2026-06-14 15:08:00.958 | INFO     | mltrainer.trainer:report:175 - Resuming epochs from previous training at 3
2026-06-14 15:08:01.052 | INFO     | mltrainer.trainer:report:209 - Epoch 3 train 1.9388 test 1.7912 metric ['0.3453']
100%|██████████| 81/81 [00:02<00:00, 28.68it/s]
2026-06-14 15:08:04.049 | INFO     | mltrainer.trainer:report:209 - Epoch 4 train 1.6922 test 1.5968 metric ['0.3781']
100%|██████████| 81/81 [00:01<00:00, 80.74it/s]
2026-06-14 15:08:05.181 | INFO     | mltrainer.trainer:report:209 - Epoch 5 train 1.5194 test 1.3819 metric ['0.5141']
100%|██████████| 3/3 [00:05<00:00,  1.74s/it]


In [66]:
mlflow.end_run()

# RNN experiments
This notebook adds GRU, LSTM and Conv1D+RNN model variants and a small sanity-run cell. Run sequentially in a kernel with `torch` installed.

In [67]:
# Imports and data
import torch
import torch.optim as optim
from torch import nn
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer import Trainer, TrainerSettings, ReportTypes, metrics
from mltrainer.preprocessors import BasePreprocessor

factory = DatasetFactoryProvider.create_factory(DatasetType.FASHION)
pre = BasePreprocessor()
streamers = factory.create_datastreamer(batchsize=64, preprocessor=pre)
train = streamers['train'].stream()
valid = streamers['valid'].stream()

accuracy = metrics.Accuracy()
loss_fn = nn.CrossEntropyLoss()

2026-06-14 15:08:05.206 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at /home/fblaak/.cache/mads_datasets/fashionmnist
2026-06-14 15:08:05.207 | INFO     | mads_datasets.base:download_data:124 - File already exists at /home/fblaak/.cache/mads_datasets/fashionmnist/fashionmnist.pt


In [68]:
# Model variants: simple GRU, LSTM, Conv1D+GRU
class GRUModel(nn.Module):
    def __init__(self, input_size=28, hidden=128, num_layers=1, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.rnn = nn.GRU(input_size, hidden, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden, num_classes)
    def forward(self, x):
        # x: (B, 1, 28, 28) -> (B, 28, 28)
        x = x.view(x.size(0), 28, 28)
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out)

class LSTMModel(nn.Module):
    def __init__(self, input_size=28, hidden=128, num_layers=1, num_classes=10):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden, num_classes)
    def forward(self, x):
        x = x.view(x.size(0), 28, 28)
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out)

class Conv1DRNNModel(nn.Module):
    def __init__(self, input_size=28, hidden=128, num_layers=1, num_classes=10):
        super().__init__()
        self.conv = nn.Conv1d(in_channels=28, out_channels=28, kernel_size=3, padding=1)
        self.rnn = nn.GRU(28, hidden, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden, num_classes)
    def forward(self, x):
        # x: (B,1,28,28) -> (B,28,28) -> (B,28,28) conv1d expects (B, C, L)
        x = x.view(x.size(0), 28, 28)
        x = self.conv(x)
        x = x.permute(0, 2, 1)  # (B, L, C)
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out)


In [69]:
# Runner (sanity-run): three models, 1 epoch, small steps
from datetime import datetime
models = [GRUModel(), LSTMModel(), Conv1DRNNModel()]
names = ['GRU','LSTM','Conv1D+GRU']
for m, n in zip(models, names):
    try:
        m.experiment = f"sanity_rnn_{n}"
        settings = TrainerSettings(
            epochs=1,
            metrics=[accuracy],
            logdir=str(LOGDIR),
            train_steps=50,
            valid_steps=50,
            reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
        )
        trainer = Trainer(
            model=m,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=train,
            validdataloader=valid,
            scheduler=None,
            device=device,
        )
        print(datetime.now(), 'Starting sanity run:', n)
        trainer.loop()
        print(datetime.now(), 'Finished sanity run:', n)
    except Exception as e:
        print('Sanity run failed for', n, e)


2026-06-14 15:08:05.270 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to 3-hypertuning-rnn/modellogs/20260614-150805
2026-06-14 15:08:05.270 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


2026-06-14 15:08:05.273771 Starting sanity run: GRU


100%|██████████| 50/50 [00:01<00:00, 44.53it/s]
2026-06-14 15:08:06.757 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.8548 test 1.3599 metric ['0.4544']
100%|██████████| 1/1 [00:01<00:00,  1.48s/it]
2026-06-14 15:08:06.762 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to 3-hypertuning-rnn/modellogs/20260614-150806
2026-06-14 15:08:06.763 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


2026-06-14 15:08:06.761488 Finished sanity run: GRU
2026-06-14 15:08:06.765856 Starting sanity run: LSTM


100%|██████████| 50/50 [00:01<00:00, 38.74it/s]
2026-06-14 15:08:08.422 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.8511 test 1.2851 metric ['0.4934']
100%|██████████| 1/1 [00:01<00:00,  1.66s/it]
2026-06-14 15:08:08.425 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to 3-hypertuning-rnn/modellogs/20260614-150808
2026-06-14 15:08:08.426 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


2026-06-14 15:08:08.425053 Finished sanity run: LSTM
2026-06-14 15:08:08.428115 Starting sanity run: Conv1D+GRU


100%|██████████| 50/50 [00:01<00:00, 42.76it/s]
2026-06-14 15:08:09.978 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.8374 test 1.3699 metric ['0.3978']
100%|██████████| 1/1 [00:01<00:00,  1.55s/it]

2026-06-14 15:08:09.980267 Finished sanity run: Conv1D+GRU


In [70]:
# Diagnostic: verify LOGDIR, TrainerSettings and that Trainer writes reports
from pathlib import Path
print('LOGDIR:', LOGDIR)
print('LOGDIR exists:', Path(LOGDIR).exists())
print('LOGDIR contents before run:', sorted([p.name for p in Path(LOGDIR).iterdir()]) if Path(LOGDIR).exists() else [])

print('TrainerSettings preview:')
tmp_settings = TrainerSettings(
    epochs=1,
    metrics=[accuracy],
    logdir=str(LOGDIR),
    train_steps=1,
    valid_steps=1,
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD],
)
print('  logdir ->', tmp_settings.logdir)

# Minimal debug run using the first model (won't take long)
m = GRUModel()
m.experiment = 'debug_write_logs'
debug_trainer = Trainer(
    model=m,
    settings=tmp_settings,
    loss_fn=loss_fn,
    optimizer=optim.Adam,
    traindataloader=train,
    validdataloader=valid,
    scheduler=None,
    device=device,
)
print('Starting debug trainer.loop()')
try:
    debug_trainer.loop()
except Exception as e:
    print('Debug trainer failed:', e)
print('Finished. LOGDIR contents after run:', sorted([p.name for p in Path(LOGDIR).iterdir()]))
print('List files in the new run folder:')
new_runs = sorted([p for p in Path(LOGDIR).iterdir() if p.is_dir()])
if new_runs:
    latest = new_runs[-1]
    print(latest)
    print(sorted([f.name for f in latest.iterdir()]))
else:
    print('No run folders found in LOGDIR after run')


2026-06-14 15:15:09.046 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to 3-hypertuning-rnn/modellogs/20260614-151509
2026-06-14 15:15:09.049 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


LOGDIR: 3-hypertuning-rnn/modellogs
LOGDIR exists: True
LOGDIR contents before run: ['20260614-141216', '20260614-142024', '20260614-150755', '20260614-150805', '20260614-150806', '20260614-150808']
TrainerSettings preview:
  logdir -> 3-hypertuning-rnn/modellogs
Starting debug trainer.loop()


100%|██████████| 1/1 [00:00<00:00, 11.55it/s]
2026-06-14 15:15:09.166 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 2.3128 test 2.2964 metric ['0.0938']
100%|██████████| 1/1 [00:00<00:00,  8.67it/s]

Finished. LOGDIR contents after run: ['20260614-141216', '20260614-142024', '20260614-150755', '20260614-150805', '20260614-150806', '20260614-150808', '20260614-151509']
List files in the new run folder:
3-hypertuning-rnn/modellogs/20260614-151509
['events.out.tfevents.1781442909.uos3fjblaak.uos3-han.src.surf-hosted.nl.102165.9', 'model.toml', 'settings.toml']


In [71]:
# Re-run full sanity-run for all three RNN variants (1 epoch each)
from datetime import datetime
models = [GRUModel(), LSTMModel(), Conv1DRNNModel()]
names = ['GRU','LSTM','Conv1D+GRU']
for m, n in zip(models, names):
    try:
        m.experiment = f"sanity_rnn_{n}"
        settings = TrainerSettings(
            epochs=1,
            metrics=[accuracy],
            logdir=str(LOGDIR),
            train_steps=50,
            valid_steps=50,
            reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML],
        )
        trainer = Trainer(
            model=m,
            settings=settings,
            loss_fn=loss_fn,
            optimizer=optim.Adam,
            traindataloader=train,
            validdataloader=valid,
            scheduler=None,
            device=device,
        )
        print(datetime.now(), 'Starting sanity run:', n)
        trainer.loop()
        print(datetime.now(), 'Finished sanity run:', n)
    except Exception as e:
        print('Sanity run failed for', n, e)


2026-06-14 15:21:18.603 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to 3-hypertuning-rnn/modellogs/20260614-152118
2026-06-14 15:21:18.604 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


2026-06-14 15:21:18.606216 Starting sanity run: GRU


100%|██████████| 50/50 [00:01<00:00, 41.98it/s]
2026-06-14 15:21:20.168 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.8143 test 1.3766 metric ['0.4306']
100%|██████████| 1/1 [00:01<00:00,  1.56s/it]
2026-06-14 15:21:20.170 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to 3-hypertuning-rnn/modellogs/20260614-152120
2026-06-14 15:21:20.173 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


2026-06-14 15:21:20.170154 Finished sanity run: GRU
2026-06-14 15:21:20.174970 Starting sanity run: LSTM


100%|██████████| 50/50 [00:00<00:00, 65.51it/s]
2026-06-14 15:21:21.294 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.8191 test 1.3431 metric ['0.5131']
100%|██████████| 1/1 [00:01<00:00,  1.12s/it]
2026-06-14 15:21:21.296 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to 3-hypertuning-rnn/modellogs/20260614-152121
2026-06-14 15:21:21.297 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


2026-06-14 15:21:21.296125 Finished sanity run: LSTM
2026-06-14 15:21:21.298526 Starting sanity run: Conv1D+GRU


100%|██████████| 50/50 [00:01<00:00, 40.93it/s]
2026-06-14 15:21:22.897 | INFO     | mltrainer.trainer:report:209 - Epoch 0 train 1.8307 test 1.3446 metric ['0.4188']
100%|██████████| 1/1 [00:01<00:00,  1.60s/it]

2026-06-14 15:21:22.899510 Finished sanity run: Conv1D+GRU


In [72]:
# Quick Conv2D baseline: short training (3 epochs) and logs to LOGDIR
print('Starting short Conv2D baseline run (3 epochs)')
try:
    short_settings = TrainerSettings(epochs=3, metrics=[accuracy], logdir=str(LOGDIR), train_steps=train_steps, valid_steps=valid_steps, reporttypes=[ReportTypes.TENSORBOARD, ReportTypes.TOML])
    trainer_short = Trainer(model=model, settings=short_settings, loss_fn=loss_fn, optimizer=optim.Adam, traindataloader=train_stream, validdataloader=valid_stream, scheduler=None, device=device)
    trainer_short.loop()
    print('Finished Conv2D short run')
except Exception as e:
    print('Conv2D short run failed:', e)


2026-06-14 15:21:22.907 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to 3-hypertuning-rnn/modellogs/20260614-152122
2026-06-14 15:21:22.908 | INFO     | mltrainer.trainer:__init__:68 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.


Starting short Conv2D baseline run (3 epochs)


  0%|          | 0/3 [00:00<?, ?it/s]

Conv2D short run failed: GRU: Expected input to be 2D or 3D, got 4D instead
